# W&B Best Baseline Hyperparams (single recipe per model)

This notebook queries Weights & Biases and picks **one hyperparameter tuple per baseline** for use on all sessions. Tuning uses three proxy sessions (low / medium / high quality); validation scores are combined across those sessions with a configurable objective (mean, median, worst-session, or mean minus penalty for variance)—a standard way to avoid overfitting to a single session.

## 1. Setup and Configuration

Configure the W&B project, sessions, and metric to use for selecting best hyperparams.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import wandb

# Load Foundry/.env into os.environ (stdlib only; no python-dotenv)
_ENV_FILE = Path("/iopsstor/scratch/cscs/esuarez/Foundry/.env")
if _ENV_FILE.is_file():
    for _raw in _ENV_FILE.read_text().splitlines():
        _line = _raw.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _key, _, _val = _line.partition("=")
        _key = _key.strip()
        if not _key:
            continue
        _val = _val.strip().strip('"').strip("'")
        os.environ.setdefault(_key, _val)
    print(f"Loaded environment keys from {_ENV_FILE}")
else:
    print(f"Env file not found: {_ENV_FILE}")

print("Imports successful")

Loaded environment keys from /iopsstor/scratch/cscs/esuarez/Foundry/.env
Imports successful


In [2]:
# ====== CONFIGURATION PARAMETERS (edit these) ======

# W&B: set WANDB_ENTITY and WANDB_API_KEY in the shell (or your Jupyter kernel env) before starting
ENTITY = os.environ.get("WANDB_ENTITY")
PROJECT = "auditory_decoding"
GROUP = "NEUROSOFT_SINGLE_SESSION"
WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

print(
    "WANDB_ENTITY:",
    ENTITY or "(missing — export it in the environment the notebook runs in)",
)
print(
    "WANDB_API_KEY:",
    "set"
    if WANDB_API_KEY
    else "(missing — export it in the environment the notebook runs in)",
)


# Login to W&B (you'll need to have wandb installed and be logged in)
try:
    wandb.login(key=WANDB_API_KEY)
    print(f"Successfully logged in to W&B as {wandb.api.viewer()['username']}")
except Exception as e:
    print(f"Error logging in to W&B: {e}")
    print("Please run 'wandb login' in your terminal and try again.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /users/esuarez/.netrc


WANDB_ENTITY: suarezul
WANDB_API_KEY: set


wandb: Currently logged in as: suarezul to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged in to W&B as suarezul


In [3]:
# Tags that runs must include to be considered
TAGS_MUST_INCLUDE = ["hparam_tuning", "baselines", "neurosoft", "raw_data"]

# The three recording sessions to analyze (proxy for low / medium / high quality in the full bank)
SESSIONS = [
    "sub-02_ses-01_task-AcousStim_acq-LH_desc-raw",  # HPARAM TUNING session 1
    "sub-04_ses-02_task-AcousStim_acq-LH_desc-raw",  # HPARAM TUNING session 2
    "sub-07_ses-03_task-AcousStim_acq-RH_desc-raw",  # HPARAM TUNING session 3
]

# Combine validation metrics across SESSIONS to score each (baseline, hparam) candidate.
# Only candidates with a run in *every* session are considered (inner join on the grid).
#   "mean"            — maximize average validation metric (default).
#   "median"          — maximize median across sessions (robust).
#   "min"             — maximin: maximize the worst session (avoid fragile configs).
#   "mean_minus_std"  — mean − HPARAM_STABILITY_LAMBDA * std (reward consistency).
HPARAM_CROSS_SESSION_OBJECTIVE = "mean"
HPARAM_STABILITY_LAMBDA = (
    0.5  # used only when HPARAM_CROSS_SESSION_OBJECTIVE == "mean_minus_std"
)

# Metric to use for selecting best hyperparams (higher is better by default)
# Valid metrics include:
# _auroc, _balanced_acc, _cohen_kappa, _f1, _loss, _precision, _recall
METRIC = "val/neurosoft_acoustic_stim_f1"
METRIC_DIRECTION = "max"  # "max" for maximize, "min" for minimize

# Filter by run state
STATE_FILTER = "finished"  # or "crashed" if you want to include partial runs

print(f"Sessions to analyze: {len(SESSIONS)}")
print(f"Metric: {METRIC} ({METRIC_DIRECTION})")
print(
    f"Hparam cross-session objective: {HPARAM_CROSS_SESSION_OBJECTIVE}"
    + (
        f" (lambda={HPARAM_STABILITY_LAMBDA})"
        if HPARAM_CROSS_SESSION_OBJECTIVE == "mean_minus_std"
        else ""
    )
)

Sessions to analyze: 3
Metric: val/neurosoft_acoustic_stim_f1 (max)
Hparam cross-session objective: mean


## 2. Query W&B Runs

Fetch all relevant runs and extract hyperparameters, session info, and metric values.

In [4]:
# Initialize W&B API
api = wandb.Api()

# Build filter for runs
filters = {
    "group": {"$eq": GROUP},
    "state": {"$eq": STATE_FILTER},
}

# Add tag filters (all tags must be present)
tag_filters = {"tags": {"$all": TAGS_MUST_INCLUDE}}
filters.update(tag_filters)

project_path = f"{ENTITY}/{PROJECT}"
print(f"Querying runs from {project_path}...")
print(f"Filters: {filters}")

# Query all matching runs (paginate through all results)
all_runs = []
for run in api.runs(project_path, filters=filters):
    all_runs.append(run)

print(f"Total runs found: {len(all_runs)}")

# Extract data from runs
run_data = []

for run in all_runs:
    try:
        # Extract session (recording_id)
        session = None
        if "data" in run.config and "recording_ids" in run.config["data"]:
            recording_ids = run.config["data"]["recording_ids"]
            if isinstance(recording_ids, list) and len(recording_ids) > 0:
                session = recording_ids[0]

        # Extract baseline model
        baseline = None
        if (
            "hydra" in run.config
            and "runtime" in run.config["hydra"]
            and "choices" in run.config["hydra"]["runtime"]
        ):
            baseline = run.config["hydra"]["runtime"]["choices"].get("model")

        # Fallback: try to extract from run name or tags
        if not baseline:
            # Look for model tags
            for tag in run.tags:
                if tag in [
                    "linear",
                    "mlp",
                    "gru",
                    "shallowcnn",
                    "temporalcnn",
                    "eegnet",
                ]:
                    baseline = tag
                    break

        # Extract metric value. W&B summary for metrics with a defined goal
        # (max/min/last) is stored as a sub-dict like {"max": 0.42}, so unwrap it.
        metric_value = run.summary.get(METRIC)
        if hasattr(metric_value, "keys"):
            metric_value = (
                metric_value.get("max")
                if METRIC_DIRECTION == "max"
                else metric_value.get("min")
            )
            if metric_value is None:
                continue
        try:
            metric_value = float(metric_value)
        except (TypeError, ValueError):
            continue

        # Extract hyperparameters
        batch_size = None
        learning_rate = None
        weight_decay = None

        if "hyperparameters" in run.config:
            hparams = run.config["hyperparameters"]
            batch_size = hparams.get("batch_size")
            learning_rate = hparams.get("learning_rate")
            weight_decay = hparams.get("weight_decay")

        # Skip if missing critical fields
        if not session or not baseline or metric_value is None:
            continue

        run_data.append(
            {
                "session": session,
                "baseline": baseline,
                "batch_size": batch_size,
                "learning_rate": learning_rate,
                "weight_decay": weight_decay,
                "metric_value": metric_value,
                "metric_name": METRIC,
                "run_id": run.id,
                "run_name": run.name,
                "run_url": run.url,
            }
        )
    except Exception as e:
        print(f"Error processing run {run.id}: {e}")

print(f"\nRuns with valid data: {len(run_data)}")

if run_data:
    df = pd.DataFrame(run_data)
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nSessions found: {sorted(df['session'].unique())}")
    print(f"Baselines found: {sorted(df['baseline'].unique())}")
else:
    print("No valid run data extracted.")
    df = pd.DataFrame()

df.to_csv("hyper-param-search.csv")

Querying runs from suarezul/auditory_decoding...
Filters: {'group': {'$eq': 'NEUROSOFT_SINGLE_SESSION'}, 'state': {'$eq': 'finished'}, 'tags': {'$all': ['hparam_tuning', 'baselines', 'neurosoft', 'raw_data']}}
Total runs found: 306

Runs with valid data: 306

DataFrame shape: (306, 10)

Sessions found: ['sub-02_ses-01_task-AcousStim_acq-LH_desc-raw', 'sub-04_ses-02_task-AcousStim_acq-LH_desc-raw', 'sub-07_ses-03_task-AcousStim_acq-RH_desc-raw']
Baselines found: ['eegnet', 'gru', 'linear', 'mlp', 'shallowcnn', 'temporalcnn']


## 3. Filter to Target Sessions

In [5]:
# Filter to only the sessions we care about
if not df.empty:
    df_filtered = df[df["session"].isin(SESSIONS)].copy()
    print(f"Runs in target sessions: {len(df_filtered)} / {len(df)}")
    print(f"\nBreakdown by session and baseline:")
    print(
        df_filtered.groupby(["session", "baseline"])
        .size()
        .unstack(fill_value=0)
    )
else:
    df_filtered = df

Runs in target sessions: 306 / 306

Breakdown by session and baseline:
baseline                                      eegnet  gru  linear  mlp  \
session                                                                  
sub-02_ses-01_task-AcousStim_acq-LH_desc-raw      18   18      12   18   
sub-04_ses-02_task-AcousStim_acq-LH_desc-raw      18   18      12   18   
sub-07_ses-03_task-AcousStim_acq-RH_desc-raw      18   18      12   18   

baseline                                      shallowcnn  temporalcnn  
session                                                                
sub-02_ses-01_task-AcousStim_acq-LH_desc-raw          18           18  
sub-04_ses-02_task-AcousStim_acq-LH_desc-raw          18           18  
sub-07_ses-03_task-AcousStim_acq-RH_desc-raw          18           18  


## 4. Select best hyperparameters per baseline (across sessions)

For each baseline, we score every hyperparameter tuple `(batch_size, learning_rate, weight_decay)` that was **evaluated on all tuning sessions**. If the same tuple appears more than once in one session (e.g. reruns), we average the metric within that session first. We then aggregate across sessions using `HPARAM_CROSS_SESSION_OBJECTIVE` and pick the tuple with the best score. This yields **one recipe per baseline** for downstream experiments.

In [6]:
# One hyperparameter recipe per baseline: aggregate validation across tuning sessions
import ast

if not df_filtered.empty:

    def _to_scalar(v):
        if isinstance(v, dict):
            return v.get("max") if METRIC_DIRECTION == "max" else v.get("min")
        if isinstance(v, str):
            try:
                parsed = ast.literal_eval(v)
            except (ValueError, SyntaxError):
                return pd.to_numeric(v, errors="coerce")
            if isinstance(parsed, dict):
                return (
                    parsed.get("max")
                    if METRIC_DIRECTION == "max"
                    else parsed.get("min")
                )
            return parsed
        return v

    df_work = df_filtered.copy()
    df_work["metric_value"] = pd.to_numeric(
        df_work["metric_value"].map(_to_scalar), errors="coerce"
    )
    df_work = df_work.dropna(subset=["metric_value"])

    hparam_keys = ["batch_size", "learning_rate", "weight_decay"]
    # Average within (session, baseline, hparams) in case of duplicate runs
    inner = df_work.groupby(
        ["session", "baseline", *hparam_keys], as_index=False
    )["metric_value"].mean()

    pt = inner.pivot_table(
        index=["baseline", *hparam_keys],
        columns="session",
        values="metric_value",
        aggfunc="first",
    )
    pt = pt.reindex(columns=list(SESSIONS))
    complete = pt.dropna(how="any")
    if complete.empty:
        print(
            "No hyperparameter tuple appears in all tuning sessions; check data."
        )
        best_hparams = pd.DataFrame()
        per_session_runs = pd.DataFrame()
    else:
        axis = 1
        mean_across = complete.mean(axis=axis)
        std_across = complete.std(axis=axis, ddof=0)
        median_across = complete.median(axis=axis)
        min_across = complete.min(axis=axis)
        max_across = complete.max(axis=axis)

        mode = HPARAM_CROSS_SESSION_OBJECTIVE
        if mode not in {"mean", "median", "min", "mean_minus_std"}:
            raise ValueError(
                f"Unknown HPARAM_CROSS_SESSION_OBJECTIVE={mode!r}; "
                "use 'mean', 'median', 'min', or 'mean_minus_std'."
            )

        if METRIC_DIRECTION == "max":
            if mode == "mean":
                selection_score = mean_across
            elif mode == "median":
                selection_score = median_across
            elif mode == "min":
                selection_score = min_across
            else:
                selection_score = (
                    mean_across - HPARAM_STABILITY_LAMBDA * std_across
                )
        else:
            # Higher selection_score = better (lower loss / cost)
            if mode == "mean":
                selection_score = -mean_across
            elif mode == "median":
                selection_score = -median_across
            elif mode == "min":
                selection_score = -max_across
            else:
                selection_score = -(
                    mean_across + HPARAM_STABILITY_LAMBDA * std_across
                )

        winners = []
        for bl in sorted(complete.index.get_level_values(0).unique()):
            mask = complete.index.get_level_values(0) == bl
            sub = selection_score[mask]
            if sub.empty:
                continue
            best_multi_idx = sub.idxmax()
            row_metrics = complete.loc[best_multi_idx]
            winners.append(
                {
                    "baseline": bl,
                    "batch_size": best_multi_idx[1],
                    "learning_rate": best_multi_idx[2],
                    "weight_decay": best_multi_idx[3],
                    "selection_objective": mode,
                    "selection_score": float(sub.loc[best_multi_idx]),
                    "mean_metric": float(mean_across.loc[best_multi_idx]),
                    "std_metric": float(std_across.loc[best_multi_idx]),
                    "min_metric": float(min_across.loc[best_multi_idx]),
                    "max_metric": float(max_across.loc[best_multi_idx]),
                    **{
                        f"metric__{ses}": float(row_metrics[ses])
                        for ses in SESSIONS
                    },
                }
            )

        best_hparams = pd.DataFrame(winners)

        # Resolve one W&B run per (session, winning config) — prefer the run with best metric in that session
        run_rows = []
        for _, w in best_hparams.iterrows():
            for ses in SESSIONS:
                cand = df_work[
                    (df_work["baseline"] == w["baseline"])
                    & (df_work["session"] == ses)
                    & (df_work["batch_size"] == w["batch_size"])
                    & (df_work["learning_rate"] == w["learning_rate"])
                    & (df_work["weight_decay"] == w["weight_decay"])
                ]
                if cand.empty:
                    continue
                best_run_idx = (
                    cand["metric_value"].idxmax()
                    if METRIC_DIRECTION == "max"
                    else cand["metric_value"].idxmin()
                )
                r = df_work.loc[best_run_idx]
                run_rows.append(
                    {
                        "baseline": w["baseline"],
                        "session": ses,
                        "batch_size": w["batch_size"],
                        "learning_rate": w["learning_rate"],
                        "weight_decay": w["weight_decay"],
                        "metric_value": r["metric_value"],
                        "run_id": r["run_id"],
                        "run_name": r.get("run_name"),
                        "run_url": r["run_url"],
                    }
                )
        per_session_runs = pd.DataFrame(run_rows)

        print(
            f"Cross-session objective: {mode} (metric direction: {METRIC_DIRECTION}); "
            f"lambda={HPARAM_STABILITY_LAMBDA if mode == 'mean_minus_std' else 'n/a'}"
        )
        print(
            f"Candidates with all {len(SESSIONS)} sessions: {len(complete)} (baseline × hparam rows)"
        )
        print(f"Best hyperparams per baseline: {len(best_hparams)} rows\n")
        disp = best_hparams[
            [
                "baseline",
                "batch_size",
                "learning_rate",
                "weight_decay",
                "selection_score",
                "mean_metric",
                "std_metric",
                "min_metric",
            ]
        ]
        print(disp.to_string(index=False))
else:
    print("No data to process.")
    best_hparams = pd.DataFrame()
    per_session_runs = pd.DataFrame()

Cross-session objective: mean (metric direction: max); lambda=n/a
Candidates with all 3 sessions: 102 (baseline × hparam rows)
Best hyperparams per baseline: 6 rows

   baseline  batch_size  learning_rate  weight_decay  selection_score  mean_metric  std_metric  min_metric
     eegnet          16         0.0150        0.0180         0.558638     0.558638    0.285350    0.238068
        gru          16         0.0015        0.0180         0.634990     0.634990    0.269981    0.295933
     linear          16         0.1500        0.0180         0.281455     0.281455    0.160406    0.089406
        mlp          16         0.0150        0.1800         0.302105     0.302105    0.154163    0.084798
 shallowcnn          16         0.0015        0.0018         0.492025     0.492025    0.261751    0.206958
temporalcnn          16         0.0015        0.0180         0.261144     0.261144    0.155362    0.059439


## 5. Chosen recipe per baseline (with per-session validation)

For each baseline: chosen `(batch_size, learning_rate, weight_decay)`, cross-session aggregate score, and the validation metric on each tuning session (proxy for low / medium / high recording quality).

In [7]:
# Human-readable summary: one row per baseline
if not best_hparams.empty:
    print("\n" + "=" * 100)
    print(
        "Chosen hyperparameters per baseline (single recipe for all sessions)"
    )
    print("=" * 100)
    print("Tuning sessions (in order):")
    for ses in SESSIONS:
        print(f"  - {ses}")

    for _, row in best_hparams.sort_values("baseline").iterrows():
        hstr = f"BS={row['batch_size']}, LR={row['learning_rate']:.4f}, WD={row['weight_decay']:.4f}"
        print(f"\n{row['baseline']}: {hstr}")
        print(
            f"  selection: {row['selection_objective']} → score={row['selection_score']:.4f} | "
            f"mean={row['mean_metric']:.4f} std={row['std_metric']:.4f} "
            f"min={row['min_metric']:.4f} max={row['max_metric']:.4f}"
        )
        if not per_session_runs.empty:
            _order = {ses: i for i, ses in enumerate(SESSIONS)}
            pr = per_session_runs[
                per_session_runs["baseline"] == row["baseline"]
            ].copy()
            pr["_sess_order"] = pr["session"].map(_order)
            pr = pr.sort_values("_sess_order").drop(columns=["_sess_order"])
            for _, rr in pr.iterrows():
                print(
                    f"    {rr['session']}: metric={rr['metric_value']:.4f}  {rr['run_url']}"
                )
else:
    print("No best hyperparameters to display.")


Chosen hyperparameters per baseline (single recipe for all sessions)
Tuning sessions (in order):
  - sub-02_ses-01_task-AcousStim_acq-LH_desc-raw
  - sub-04_ses-02_task-AcousStim_acq-LH_desc-raw
  - sub-07_ses-03_task-AcousStim_acq-RH_desc-raw

eegnet: BS=16, LR=0.0150, WD=0.0180
  selection: mean → score=0.5586 | mean=0.5586 std=0.2854 min=0.2381 max=0.9312
    sub-02_ses-01_task-AcousStim_acq-LH_desc-raw: metric=0.5066  https://wandb.ai/suarezul/auditory_decoding/runs/tmgqj1iv
    sub-04_ses-02_task-AcousStim_acq-LH_desc-raw: metric=0.2381  https://wandb.ai/suarezul/auditory_decoding/runs/qo3090gh
    sub-07_ses-03_task-AcousStim_acq-RH_desc-raw: metric=0.9312  https://wandb.ai/suarezul/auditory_decoding/runs/z8e7e7qf

gru: BS=16, LR=0.0015, WD=0.0180
  selection: mean → score=0.6350 | mean=0.6350 std=0.2700 min=0.2959 max=0.9566
    sub-02_ses-01_task-AcousStim_acq-LH_desc-raw: metric=0.6525  https://wandb.ai/suarezul/auditory_decoding/runs/378gaeas
    sub-04_ses-02_task-AcousStim

## 6. Export to CSV

In [8]:
# # Export one row per baseline + audit columns for each tuning session
# if not best_hparams.empty:
#     export_rows = []
#     for _, w in best_hparams.iterrows():
#         row = {
#             "baseline": w["baseline"],
#             "batch_size": w["batch_size"],
#             "learning_rate": w["learning_rate"],
#             "weight_decay": w["weight_decay"],
#             "metric_name": METRIC,
#             "selection_objective": w["selection_objective"],
#             "selection_score": w["selection_score"],
#             "mean_metric": w["mean_metric"],
#             "std_metric": w["std_metric"],
#             "min_metric": w["min_metric"],
#             "max_metric": w["max_metric"],
#         }
#         for i, ses in enumerate(SESSIONS):
#             row[f"tuning_session_{i}"] = ses
#             if not per_session_runs.empty:
#                 hit = per_session_runs[
#                     (per_session_runs["baseline"] == w["baseline"])
#                     & (per_session_runs["session"] == ses)
#                 ]
#                 if not hit.empty:
#                     rr = hit.iloc[0]
#                     row[f"metric_sess{i}"] = rr["metric_value"]
#                     row[f"wandb_run_id_sess{i}"] = rr["run_id"]
#                     row[f"wandb_url_sess{i}"] = rr["run_url"]
#         export_rows.append(row)
#     export_df = pd.DataFrame(export_rows)
#     csv_path = Path("best_baseline_hparams.csv")
#     export_df.to_csv(csv_path, index=False)
#     print(f"Exported to: {csv_path.absolute()}")
#     print("\nPreview:")
#     print(export_df.to_string())
# else:
#     print("No data to export.")

## 7. Summary (single recipe per baseline + W&B links)

In [9]:
# Final summary: one recipe per baseline + W&B run for each tuning session
if not best_hparams.empty:
    print("\n" + "=" * 100)
    print(
        "Summary: cross-session baseline hyperparameters (single recipe per model)"
    )
    print("=" * 100)
    print(f"\nMetric: {METRIC} ({METRIC_DIRECTION})")
    print(f"Cross-session objective: {HPARAM_CROSS_SESSION_OBJECTIVE}")
    print(f"Tuning sessions ({len(SESSIONS)}): {SESSIONS}")
    print(f"Baselines: {best_hparams['baseline'].nunique()}")
    print(
        "\nUse the same batch_size / learning_rate / weight_decay for every session in downstream runs.\n"
    )

    for _, row in best_hparams.sort_values("baseline").iterrows():
        print(f"{row['baseline']}:")
        print(f"  batch_size={row['batch_size']}")
        print(f"  learning_rate={row['learning_rate']}")
        print(f"  weight_decay={row['weight_decay']}")
        print(
            f"  aggregate: mean={row['mean_metric']:.4f} std={row['std_metric']:.4f} "
            f"min={row['min_metric']:.4f} max={row['max_metric']:.4f} "
            f"(selection_score={row['selection_score']:.4f}, rule={row['selection_objective']})"
        )
        if not per_session_runs.empty:
            _order = {ses: i for i, ses in enumerate(SESSIONS)}
            pr = per_session_runs[
                per_session_runs["baseline"] == row["baseline"]
            ].copy()
            pr["_sess_order"] = pr["session"].map(_order)
            pr = pr.sort_values("_sess_order").drop(columns=["_sess_order"])
            for _, rr in pr.iterrows():
                print(f"  [{rr['session']}] {METRIC}={rr['metric_value']:.4f}")
                print(f"    W&B: {rr['run_url']}")
        print()
else:
    print("No data to summarize.")


Summary: cross-session baseline hyperparameters (single recipe per model)

Metric: val/neurosoft_acoustic_stim_f1 (max)
Cross-session objective: mean
Tuning sessions (3): ['sub-02_ses-01_task-AcousStim_acq-LH_desc-raw', 'sub-04_ses-02_task-AcousStim_acq-LH_desc-raw', 'sub-07_ses-03_task-AcousStim_acq-RH_desc-raw']
Baselines: 6

Use the same batch_size / learning_rate / weight_decay for every session in downstream runs.

eegnet:
  batch_size=16
  learning_rate=0.015
  weight_decay=0.018
  aggregate: mean=0.5586 std=0.2854 min=0.2381 max=0.9312 (selection_score=0.5586, rule=mean)
  [sub-02_ses-01_task-AcousStim_acq-LH_desc-raw] val/neurosoft_acoustic_stim_f1=0.5066
    W&B: https://wandb.ai/suarezul/auditory_decoding/runs/tmgqj1iv
  [sub-04_ses-02_task-AcousStim_acq-LH_desc-raw] val/neurosoft_acoustic_stim_f1=0.2381
    W&B: https://wandb.ai/suarezul/auditory_decoding/runs/qo3090gh
  [sub-07_ses-03_task-AcousStim_acq-RH_desc-raw] val/neurosoft_acoustic_stim_f1=0.9312
    W&B: https://wa